# Run all model build/train/eval steps

Select the Python 3.13 kernel where TensorFlow is installed. Run each cell in order to build baseline profiles, train the attack-type classifier, train the LSTM model, run evaluation, and verify artifacts.

In [ ]:
import os, sys
repo_root = os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)
print('Repo root:', repo_root)
print('Working directory:', os.getcwd())

In [ ]:
import sys
print('Python:', sys.version.replace('
', ' '))
try:
    import tensorflow as tf
    print('TensorFlow available:', tf.__version__)
    tf_available = True
except Exception as e:
    print('TensorFlow not available in this kernel:', e)
    tf_available = False

In [ ]:
import subprocess, sys, os
from importlib import import_module

def run_module_or_call(module_name, func_name=None):
    try:
        mod = import_module(module_name)
        if func_name and hasattr(mod, func_name):
            print(f'Calling {module_name}.{func_name}()')
            getattr(mod, func_name)()
            return True
        print(f'Imported {module_name} but no direct entrypoint {func_name or ''}')
    except Exception as exc:
        print(f'Import failed for {module_name}:', exc)
    try:
        print(f'Running as module: {module_name}')
        subprocess.run([sys.executable, '-m', module_name], check=True)
        return True
    except Exception as exc:
        print(f'Running module {module_name} failed:', exc)
        return False

In [ ]:
print('--- Building baseline profiles ---')
ok = False
try:
    mod = import_module('src.baseline_profiling')
    if hasattr(mod, 'create_baseline_profile_artifact'):
        print('Calling src.baseline_profiling.create_baseline_profile_artifact()')
        mod.create_baseline_profile_artifact()
        ok = True
    else:
        ok = run_module_or_call('src.baseline_profiling', None)
except Exception as exc:
    print('Baseline build failed:', exc)
    ok = False
print('Baseline step ok=', ok)

In [ ]:
print('--- Training attack-type classifier ---')
ok = run_module_or_call('src.attack_type_classifier', 'train_attack_type_classifier')
print('Attack-type training ok=', ok)

In [ ]:
print('--- Training LSTM model (if TF available) ---')
if tf_available:
    import importlib
    try:
        mod = importlib.import_module('src.lstm_sequence_model')
        if hasattr(mod, 'train_lstm_sequence_model'):
            print('Calling src.lstm_sequence_model.train_lstm_sequence_model()')
            mod.train_lstm_sequence_model()
        else:
            print('No train_lstm_sequence_model entrypoint, falling back to module')
            run_module_or_call('src.lstm_sequence_model', None)
    except Exception as exc:
        print('LSTM training failed:', exc)
else:
    print('Skipping LSTM training: TensorFlow not available in this kernel')

In [ ]:
print('--- Running evaluation script ---')
ok = run_module_or_call('src.evaluate_models', None)
print('Evaluation ok=', ok)

In [ ]:
import os, json
artifacts = [
    'trained_models/baseline_profile.pkl',
    'trained_models/attack_type_classifier.pkl',
    'trained_models/lstm_model.keras',
    'trained_models/evaluation_results.json',
]
print('--- Artifact check ---')
for p in artifacts:
    print(p, '->', os.path.exists(p))
if os.path.exists('trained_models/evaluation_results.json'):
    try:
        data = json.load(open('trained_models/evaluation_results.json', encoding='utf-8'))
        print('evaluation_results contains keys:', list(data.keys()))
    except Exception as exc:
        print('Failed reading evaluation_results.json', exc)